# Step by Step 06 - Complete Backtest
-----------------------------

This example combines all previous modules into a complete backtest.

In [ ]:
import asyncio
import datetime as dt
import numpy as np


## 1. Backtest Settings

This is a very simple factor backtest based on an index-enhancement strategy, using a medium-to-low-frequency approach:

1. Rebalancing frequency: ten days, approximately every two weeks.
2. Rebalancing limit: at most 40% turnover per rebalance.
3. Universe: CSI 300.
4. Portfolio optimization: standard mean-variance analysis.

In [ ]:
# Back test parameter settings
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)


Two price-and-volume cross-sectional factors are used:

* EMA50;
* EMV60.

An equal-weighted factor-combination model is used.

In [ ]:
# Model settings
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


Portfolio optimization controls several basic style exposures and leverage:

* Leverage: at least 80% invested in stocks, with no leverage.
* Style exposure: zero excess exposure to market capitalization, nonlinear market capitalization, and market BETA.

In [ ]:
# Constraintes settings
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


## 2. Run the Backtest

Two high-level classes help users run backtests conveniently:

* `RunningSetting`: collection of runtime settings.
* `Strategy`: strategy class.

In [ ]:
# Running settings
import tempfile
from alphamind.data import read_parquet, write_parquet
from alphamind.data.processing import factor_processing


After configuration, run the backtest simply by calling the `run` method:

In [ ]:
# Strategy
from alphamind.data.standardize import standardize
from alphamind.data.transformer import Transformer
from alphamind.data.winsorize import winsorize_normal


Results can be analyzed directly through plots, including turnover losses and excess returns.

In [ ]:
from alphamind.execution.naiveexecutor import NaiveExecutor
from alphamind.formula import CSRank
from alphamind.portfolio.rankbuilder import rank_build


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    input_path = f"{directory}/input.parquet"
    output_path = f"{directory}/returns.parquet"
    write_parquet(panel, input_path)
    loaded = read_parquet(input_path)
    scores = await asyncio.to_thread(
        Transformer({"score": CSRank("factor")}).transform, "code", loaded
    )
    loaded = loaded.append_column("score", scores["score"])
    executor = NaiveExecutor()
    results = []
    loaded_dates = sorted(pc.unique(loaded["trade_date"]).to_pylist())
    for trade_date in loaded_dates:
        section = loaded.filter(pc.equal(loaded["trade_date"], trade_date))
        signal = section["score"].to_numpy(zero_copy_only=False).reshape(-1, 1)
        processed = factor_processing(
            signal, pre_process=[winsorize_normal, standardize], post_process=[standardize]
        ).flatten()
        weights = rank_build(processed, use_rank=3).flatten().astype(float) / 3.0
        target = pa.table({"code": section["code"], "weight": weights})
        turn_over, executed = executor.execute(target)
        executor.set_current(executed)
        portfolio_return = float(weights @ section["dx"].to_numpy(zero_copy_only=False))
        results.append(pa.table({
            "trade_date": [trade_date],
            "return": [portfolio_return],
            "turn_over": [turn_over],
        }))
    backtest = pa.concat_tables(results)
    write_parquet(backtest, output_path)
    restored = read_parquet(output_path)
assert restored.num_rows == n_dates
restored
